# DAY 6: Adapter Merge, Quantization & Inference Optimization

## 🎯 Learning Objectives
In this session, you transition the trained adapter into a **standalone, production-ready model**.

By the end of this day, you will be able to execute and explain:
1. **LoRA Adapter Merging**: Fusing adapter updates $\Delta W = (\alpha/r)BA$ directly into the base weights $W_0$, creating a unified weight matrix $W_{\text{merged}}$.
2. **`merge_and_unload()`**: Removing the PEFT library runtime dependency and eliminating matrix addition overhead during inference.
3. **Inference-Time Quantization**: Comparing FP16 serving vs. 4-bit quantized serving for low-latency, memory-efficient production deployment.
4. **Benchmarking Metrics**: Measuring generation latency (ms), throughput (tokens/sec), and VRAM utilization.
5. **Standalone Model Artifacts**: Exporting standard Hugging Face Safetensors format ready for FastAPI and Docker containerization.

In [1]:
!pip install -q transformers peft accelerate safetensors

In [2]:
import os
import time
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "models/adapters/qwen-1.5b-order-extractor"
MERGED_DIR = "models/merged/qwen-1.5b-order-extractor"

print("=" * 60)
print("🚀 DAY 6: ADAPTER MERGING & INFERENCE OPTIMIZATION")
print("=" * 60)

🚀 DAY 6: ADAPTER MERGING & INFERENCE OPTIMIZATION


### Step 1: Execute `merge_and_unload()`

In [4]:
import os
import sys

# 1. Pull latest code & adapter weights from GitHub
if not os.path.exists("/content/Fine-Tuning-PoCs"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
else:
    %cd /content/Fine-Tuning-PoCs
    !git pull origin main

# 2. Change directory into project root
%cd /content/Fine-Tuning-PoCs/llm-finetuning-production

# 3. Add to Python path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 4. Fix torchao compatibility for PEFT
!pip install -q -U torchao

# 5. Verify adapter is present
adapter_cfg = "models/adapters/qwen-1.5b-order-extractor/adapter_config.json"
print("✅ Working Directory:", os.getcwd())
print("✅ Adapter Config Exists:", os.path.exists(adapter_cfg))


Cloning into 'Fine-Tuning-PoCs'...
remote: Enumerating objects: 353, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 353 (delta 0), reused 5 (delta 0), pack-reused 344 (from 1)
Receiving objects: 100% (353/353), 105.71 MiB | 23.55 MiB/s, done.
Resolving deltas: 100% (140/140), done.
Updating files: 100% (119/119), done.
/content/Fine-Tuning-PoCs/llm-finetuning-production
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 54.7 MB/s eta 0:00:00a 0:00:01
✅ Working Directory: /content/Fine-Tuning-PoCs/llm-finetuning-production
✅ Adapter Config Exists: True


In [5]:
print(f"Loading base model in FP16 for weight fusion: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

# Merge on CPU to preserve GPU VRAM
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print(f"Attaching adapter weights from {ADAPTER_DIR}...")
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

print("Executing merge_and_unload() [W_merged = W_0 + (alpha/r)*B*A]...")
merged_model = peft_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
print(f"Saving merged standalone model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("✅ Standalone merged model successfully saved!")

Loading base model in FP16 for weight fusion: Qwen/Qwen2.5-1.5B-Instruct...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Attaching adapter weights from models/adapters/qwen-1.5b-order-extractor...
Executing merge_and_unload() [W_merged = W_0 + (alpha/r)*B*A]...
Saving merged standalone model to models/merged/qwen-1.5b-order-extractor...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Standalone merged model successfully saved!


### Step 2: Benchmark Standalone Merged Inference vs Unmerged Adapter

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading merged standalone model into {device}...")
prod_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
prod_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None
)
prod_model.eval()

test_prompt = "Extract order into JSON: Emma ordered 2 ergonomic chairs for $600, deliver Friday."
msgs = [
    {"role": "system", "content": 'Extract order into JSON: {"customer": str, "quantity": int, "product": str, "amount": float, "delivery_day": str}'},
    {"role": "user", "content": test_prompt}
]

formatted = prod_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = prod_tokenizer(formatted, return_tensors="pt").to(device)

# Warmup run
with torch.no_grad():
    _ = prod_model.generate(**inputs, max_new_tokens=60, do_sample=False)

# Timed Benchmark
latencies = []
for _ in range(10):
    t0 = time.perf_counter()
    with torch.no_grad():
        out = prod_model.generate(**inputs, max_new_tokens=60, do_sample=False)
    latencies.append((time.perf_counter() - t0) * 1000)

avg_latency = sum(latencies) / len(latencies)
tokens_gen = out[0].shape[0] - inputs["input_ids"].shape[1]
throughput = (tokens_gen / (avg_latency / 1000))

print("=== INFERENCE BENCHMARK (MERGED MODEL) ===")
print(f"Avg Latency per Request : {avg_latency:.2f} ms")
print(f"Generated Tokens        : {tokens_gen} tokens")
print(f"Inference Throughput    : {throughput:.2f} tokens/second")
print("Generated Output:")
print(prod_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

Loading merged standalone model into cuda...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== INFERENCE BENCHMARK (MERGED MODEL) ===
Avg Latency per Request : 1725.82 ms
Generated Tokens        : 38 tokens
Inference Throughput    : 22.02 tokens/second
Generated Output:
{"customer": "Emma", "quantity": 2, "product": "ergonomic chairs", "amount": 600.0, "delivery_day": "Friday"}


---
## 🎓 Day 6 Interview Preparation & Concept Mastery

### 🧠 Concepts You Must Know
1. **`merge_and_unload()`**: Algebraically combines $W_0$ and $\Delta W$ so the resulting model is a standard `AutoModelForCausalLM` without any PEFT wrapper or extra matrix multiplications.
2. **SafeSerialization (Safetensors)**: Fast, memory-mapped zero-copy deserialization format that prevents arbitrary code execution vulnerabilities present in Python `pickle` (`.bin` / `.pt`).
3. **Inference vs Training Quantization**: QLoRA quantizes for training (with gradient backprop into adapters); inference quantization (like GPTQ, AWQ, or FP8/INT8) quantizes weights for execution speed and lower VRAM without any adapters.
4. **Cold Start vs Steady State Latency**: The first inference request includes CUDA kernel compilation and memory allocation overhead (warmup). Benchmarks must measure steady-state execution.

### 💼 5 Interview Questions
1. **Q1 (Beginner)**: *Why do we merge the LoRA adapter before deploying to production?*
   - **Answer**: Merging fuses $BA$ into the original weight matrices, eliminating runtime PEFT overhead and allowing deployment via standard high-performance inference servers (vLLM, TGI, TensorRT-LLM, or clean PyTorch).
2. **Q2 (Intermediate)**: *Can you merge an adapter trained with 4-bit QLoRA?*
   - **Answer**: Yes, but you must load the base model in 16-bit precision (FP16 or BF16) before attaching the adapter and calling `merge_and_unload()`. Merging directly into 4-bit quantized weights causes quantization distortion because adding continuous $BA$ to discretized 4-bit weights destroys the quantization grid.
3. **Q3 (Intermediate)**: *Why is Safetensors preferred over PyTorch `.bin` or `.pt` files?*
   - **Answer**: Safetensors provides zero-copy memory mapping (`mmap`), faster file loading times, and prevents security exploits because it only stores tensor buffers and JSON headers without executing arbitrary Python pickle bytecode.
4. **Q4 (Advanced)**: *What is the difference between AWQ, GPTQ, and GGUF quantization?*
   - **Answer**: AWQ (Activation-aware Weight Quantization) protects the top 1% salient weight channels based on activation magnitudes. GPTQ quantizes weights row-by-row using second-order Hessian error minimization. GGUF is a binary file format optimized for fast CPU/Metal/GPU execution in `llama.cpp`.
5. **Q5 (Advanced)**: *How do continuous batching and KV caching improve serving throughput?*
   - **Answer**: KV caching prevents re-computing attention keys and values for previous tokens during autoregressive generation. Continuous batching dynamically schedules incoming requests into active GPU forward passes without waiting for all batch sequences to finish.

### 🛠️ 3 Practical Questions on Your Code
1. *In your code, why was `device_map="cpu"` used during the merge step?*
2. *How did you measure generation throughput in tokens per second?*
3. *What files were generated in `models/merged/qwen-1.5b-order-extractor`?*

### 🏆 Day 6 Hands-On Challenge
> **Challenge**: Measure the disk size of the base model weights vs. the LoRA adapter vs. the merged standalone weights. Verify that the merged model size is identical to the base model size.